# Adversarial Image Synthesis — Oxford 102 Flowers

A DCGAN trained to synthesize photorealistic flower images, with analysis of adversarial training dynamics, discriminator forensics, and latent space geometry.

**Sections:** Setup → Data → Architecture → Training → Training Dynamics → Discriminator Forensics → Latent Space

## 1. Setup

In [ ]:
# Install dependencies (run once)
!pip install -q torchvision torchmetrics

In [ ]:
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.utils as vutils
from torchvision import transforms
from torch.utils.data import DataLoader

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('darkgrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Data

In [ ]:
image_size = 64
batch_size = 102

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

flowers_train = torchvision.datasets.Flowers102(
    '', split='train', transform=transform, download=True
)
dataloaders = {'train': DataLoader(flowers_train, batch_size=batch_size, shuffle=True)}
print(f'Training samples: {len(flowers_train)}')
print(f'Batches per epoch: {len(dataloaders["train"])}')

In [ ]:
# Visualize a sample of real training images
real_batch = next(iter(dataloaders['train']))
plt.figure(figsize=(10, 10))
plt.axis('off')
plt.title('Real Training Images — Oxford 102 Flowers', fontsize=13, fontweight='bold', pad=12)
plt.imshow(np.transpose(
    vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(),
    (1, 2, 0)
))
plt.tight_layout()
plt.savefig('real_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Architecture

**DCGAN** — Radford et al. (2015). Discriminator uses strided Conv2d + LeakyReLU; Generator uses ConvTranspose2d + ReLU → Tanh. Both use BatchNorm for training stability.

In [ ]:
nc  = 3    # image channels (RGB)
nz  = 100  # latent vector size
ngf = 64   # generator feature map depth
ndf = 64   # discriminator feature map depth

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, nc, ndf):
        super().__init__()
        self.pipeline = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.pipeline(x)


class Generator(nn.Module):
    def __init__(self, nc, nz, ngf):
        super().__init__()
        self.pipeline = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.pipeline(x)


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


netD = Discriminator(nc, ndf).to(device)
netG = Generator(nc, nz, ngf).to(device)
netD.apply(weights_init)
netG.apply(weights_init)

d_params = sum(p.numel() for p in netD.parameters())
g_params = sum(p.numel() for p in netG.parameters())
print(f'Discriminator parameters: {d_params:,}')
print(f'Generator parameters:     {g_params:,}')

## 4. Training

In [ ]:
os.makedirs('generated_images', exist_ok=True)

def train_GANs(netD, netG, dataloaders, criterion, optimizerD, optimizerG, num_epochs=250):
    since = time.time()
    training_curves = {'D': [], 'G': [], 'D_x': [], 'D_Gz': []}
    fixed_noise = torch.randn(64, nz, 1, 1, device=device)

    for epoch in range(num_epochs):
        for i, (real_imgs, _) in enumerate(dataloaders['train']):
            real_imgs = real_imgs.to(device)
            b_size = real_imgs.size(0)

            # --- Train Discriminator ---
            optimizerD.zero_grad()
            label_real = torch.full((b_size,), 1.0, device=device)
            label_fake = torch.full((b_size,), 0.0, device=device)

            out_real = netD(real_imgs).view(-1)
            loss_real = criterion(out_real, label_real)
            loss_real.backward()
            D_x = out_real.mean().item()

            noise = torch.randn(b_size, nz, 1, 1, device=device)
            fake = netG(noise)
            out_fake = netD(fake.detach()).view(-1)
            loss_fake = criterion(out_fake, label_fake)
            loss_fake.backward()
            D_Gz1 = out_fake.mean().item()
            errD = loss_real + loss_fake
            optimizerD.step()

            # --- Train Generator ---
            optimizerG.zero_grad()
            out_fake2 = netD(fake).view(-1)
            errG = criterion(out_fake2, label_real)  # generator wants D to output 1
            errG.backward()
            D_Gz2 = out_fake2.mean().item()
            optimizerG.step()

            training_curves['D'].append(errD.item())
            training_curves['G'].append(errG.item())
            training_curves['D_x'].append(D_x)
            training_curves['D_Gz'].append(D_Gz2)

        # Save checkpoint image every epoch
        with torch.no_grad():
            fake_fixed = netG(fixed_noise).detach().cpu()
        grid = vutils.make_grid(fake_fixed, padding=2, normalize=True)
        plt.figure(figsize=(8, 8))
        plt.axis('off')
        plt.title(f'Epoch {epoch + 1}')
        plt.imshow(np.transpose(grid, (1, 2, 0)))
        plt.savefig(f'generated_images/epoch_{epoch + 1}.png', bbox_inches='tight')
        plt.close()

        if (epoch + 1) % 25 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1:>3}/{num_epochs}]  '
                  f'D: {errD.item():.4f}  G: {errG.item():.4f}  '
                  f'D(x): {D_x:.4f}  D(G(z)): {D_Gz2:.4f}')

    elapsed = time.time() - since
    print(f'\nTraining complete: {elapsed // 60:.0f}m {elapsed % 60:.0f}s')
    return netD, netG, training_curves

In [ ]:
num_epochs = 250
criterion  = nn.BCELoss()
optimizerD = torch.optim.Adam(netD.parameters(), lr=0.0001, betas=(0.5, 0.999))
optimizerG = torch.optim.Adam(netG.parameters(), lr=0.0001, betas=(0.5, 0.999))

netD, netG, training_curves = train_GANs(
    netD, netG, dataloaders, criterion, optimizerD, optimizerG, num_epochs=num_epochs
)

# Persist models
torch.save(netG.state_dict(), 'generator.pth')
torch.save(netD.state_dict(), 'discriminator.pth')
print('Models saved.')

## 5. Training Dynamics

The loss curves expose the adversarial game: early training shows the Discriminator dominating (D Loss collapses, G Loss spikes), with recovery as the Generator adapts.

In [ ]:
# --- Annotated loss curves ---
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
fig.suptitle('Adversarial Training Dynamics', fontsize=14, fontweight='bold', y=0.98)

iters = range(len(training_curves['G']))

# Generator loss
axes[0].plot(iters, training_curves['G'], color='#e74c3c', linewidth=0.6, alpha=0.8)
axes[0].set_ylabel('Generator Loss', fontsize=11)

peak_g_val = max(training_curves['G'])
peak_g_idx = training_curves['G'].index(peak_g_val)
axes[0].annotate(
    f'Discriminator dominates\n(G Loss peak: {peak_g_val:.1f})',
    xy=(peak_g_idx, peak_g_val),
    xytext=(peak_g_idx + len(iters) * 0.05, peak_g_val * 0.85),
    arrowprops=dict(arrowstyle='->', color='#333'),
    fontsize=9, color='#333'
)
axes[0].text(
    len(iters) * 0.98, training_curves['G'][-1] + 0.3,
    f'Final: {training_curves["G"][-1]:.3f}',
    ha='right', fontsize=9, color='#e74c3c'
)

# Discriminator loss
axes[1].plot(iters, training_curves['D'], color='#3498db', linewidth=0.6, alpha=0.8)
axes[1].set_ylabel('Discriminator Loss', fontsize=11)
axes[1].set_xlabel('Training Iterations', fontsize=11)
axes[1].text(
    len(iters) * 0.98, training_curves['D'][-1] + 0.01,
    f'Final: {training_curves["D"][-1]:.3f}',
    ha='right', fontsize=9, color='#3498db'
)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final G Loss: {training_curves["G"][-1]:.4f}  |  Final D Loss: {training_curves["D"][-1]:.4f}')

In [ ]:
# --- Progressive training grid ---
# Show how image quality evolves across epochs
checkpoints = [1, 10, 25, 50, 75, 100, 150, 200, 250]
available = [(ep, f'generated_images/epoch_{ep}.png')
             for ep in checkpoints if os.path.exists(f'generated_images/epoch_{ep}.png')]

if available:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(n * 3, 3.5))
    if n == 1:
        axes = [axes]
    for ax, (ep, path) in zip(axes, available):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(f'Epoch {ep}', fontsize=9)
        ax.axis('off')
    fig.suptitle('Generator Progression — Noise → Flowers', fontsize=13, fontweight='bold', y=1.04)
    plt.tight_layout()
    plt.savefig('progression.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Run training first — epoch images will appear here.')

## 6. Discriminator Forensics

The trained Discriminator is a fake-image detector. Separating its score distributions on real vs. generated images quantifies how well the adversarial game resolved — a strong GAN should push both distributions toward 0.5.

In [ ]:
netD.eval()
netG.eval()

real_scores, fake_scores = [], []

with torch.no_grad():
    # Score all real images
    for imgs, _ in dataloaders['train']:
        scores = netD(imgs.to(device)).view(-1).cpu().numpy()
        real_scores.extend(scores)

    # Score an equal number of generated images
    n_fake_batches = max(1, len(real_scores) // batch_size + 1)
    for _ in range(n_fake_batches):
        noise = torch.randn(batch_size, nz, 1, 1, device=device)
        fake = netG(noise)
        scores = netD(fake).view(-1).cpu().numpy()
        fake_scores.extend(scores)

real_scores = np.array(real_scores)
fake_scores = np.array(fake_scores[:len(real_scores)])  # match sample counts

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(real_scores, bins=60, alpha=0.7, color='#2ecc71', density=True,
        label=f'Real  (n={len(real_scores)}, mean={real_scores.mean():.3f})')
ax.hist(fake_scores, bins=60, alpha=0.7, color='#e74c3c', density=True,
        label=f'Generated  (n={len(fake_scores)}, mean={fake_scores.mean():.3f})')
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.2, label='Decision boundary (0.5)')
ax.set_xlabel('Discriminator Confidence Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Discriminator Forensics — Real vs. Generated Score Distributions', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('discriminator_forensics.png', dpi=150, bbox_inches='tight')
plt.show()

overlap = np.mean((real_scores > 0.4) & (real_scores < 0.6)) + \
          np.mean((fake_scores > 0.4) & (fake_scores < 0.6))
print(f'Real images  — mean: {real_scores.mean():.4f}, std: {real_scores.std():.4f}')
print(f'Fake images  — mean: {fake_scores.mean():.4f}, std: {fake_scores.std():.4f}')
print(f'Samples in ambiguous zone (0.4–0.6): {overlap/2:.1%}')
print(f'Separability: if both means ~0.5, adversarial equilibrium reached.')

## 7. Latent Space Exploration

A well-trained Generator maps a smooth latent space — nearby z vectors should produce visually similar flowers. Spherical interpolation (slerp) between two random z vectors reveals whether the model learned a continuous, meaningful representation or patchy noise.

In [ ]:
def slerp(z1, z2, t):
    """Spherical linear interpolation in latent space."""
    z1_n = z1 / z1.norm(dim=1, keepdim=True)
    z2_n = z2 / z2.norm(dim=1, keepdim=True)
    omega = torch.acos((z1_n * z2_n).sum(dim=1, keepdim=True).clamp(-1, 1))
    sin_omega = torch.sin(omega)
    # Fallback to linear when vectors are nearly parallel
    return torch.where(
        sin_omega < 1e-6,
        (1 - t) * z1 + t * z2,
        (torch.sin((1 - t) * omega) / sin_omega) * z1 +
        (torch.sin(t * omega) / sin_omega) * z2
    )


n_steps = 10
n_rows  = 4  # 4 independent z1→z2 walks

netG.eval()
fig, axes = plt.subplots(n_rows, n_steps, figsize=(n_steps * 2, n_rows * 2.2))
fig.suptitle('Latent Space Interpolation — Spherical (z₁ → z₂)', fontsize=13, fontweight='bold', y=1.02)

with torch.no_grad():
    for row in range(n_rows):
        z1 = torch.randn(1, nz, device=device)
        z2 = torch.randn(1, nz, device=device)
        for col in range(n_steps):
            t = col / (n_steps - 1)
            z = slerp(z1, z2, t).view(1, nz, 1, 1)
            img = netG(z).squeeze(0).cpu().permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            axes[row][col].imshow(img)
            axes[row][col].axis('off')
            if row == 0:
                if col == 0:
                    axes[row][col].set_title('z₁', fontsize=8)
                elif col == n_steps - 1:
                    axes[row][col].set_title('z₂', fontsize=8)

plt.tight_layout()
plt.savefig('latent_interpolation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Smooth transitions indicate a well-structured latent space.')
print('Abrupt jumps or texture artifacts indicate mode collapse or insufficient training.')

In [ ]:
# --- Final comparison: real vs. generated ---
real_batch = next(iter(dataloaders['train']))

with torch.no_grad():
    fake_final = netG(torch.randn(64, nz, 1, 1, device=device)).detach().cpu()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
ax1.axis('off')
ax1.set_title('Real Images', fontsize=13, fontweight='bold')
ax1.imshow(np.transpose(
    vutils.make_grid(real_batch[0].to(device)[:64], padding=4, normalize=True).cpu(), (1, 2, 0)
))
ax2.axis('off')
ax2.set_title('Generated Images', fontsize=13, fontweight='bold')
ax2.imshow(np.transpose(
    vutils.make_grid(fake_final, padding=4, normalize=True), (1, 2, 0)
))
plt.suptitle(f'Oxford 102 Flowers — Real vs. GAN-Generated ({num_epochs} epochs)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('real_vs_generated.png', dpi=150, bbox_inches='tight')
plt.show()